# Activity: Fun with Iterative Linear Algebraic Equation (LAE) Solvers
In this activity, let's explore the properties and performance of three iterative linear algebraic solvers: the Jacobi method, the Gauss-Seidel method, and the Successive Over-Relaxation (SOR) method. 

In this activity we will:
* __Task 1__: Generate random diagonally dominant system matrices $\mathbf{A}$ and right-hand-side vectors $\mathbf{b}$ of a specified dimension. We'll use these test matrices for benchmarking studies later in the activity.
* __Task 2__: Solve the LAEs using the Jacobi, Gauss-Seidel methods and SOR methods. In this task, we'll solve our system of random linear algebraic equations using the [Jacobi](https://en.wikipedia.org/wiki/Jacobi_method) and [Gauss-Seidel](https://en.wikipedia.org/wiki/Gauss%E2%80%93Seidel_method) methods
* __Task 3__: In this task, we'll compare the runtime performance of the different iterative approaches against the built-in method implemented by [the LinearAlgebra.jl package](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/) included with Julia [using the `BenchmarkTools.jl` package.](https://github.com/JuliaCI/BenchmarkTools.jl)

This should be fun, so let's go!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

The [include command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

In [1]:
include(joinpath(@__DIR__, "Include.jl"));

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types and data used in this material. 

## Task 1: Build random diagonally dominant matrices
In this task, we'll generate a random system matrix $\mathbf{A}$ that is diagonally dominant and a random right-hand side vector $\mathbf{b}$. 


> __Diagonal dominance__ is a matrix property where the absolute value of the diagonal element of each row is greater than the sum of the absolute values of the other elements in that row.  Diagonal dominance is a sufficient (but not necessary) condition for convergence of iterative methods, however, this condition says nothing about the rate of convergence.

A diagonally dominant system matrix $\mathbf{A}$ has the feature:
$$
\begin{equation*}
\sum_{j=1,i}^{n}\lvert{a_{ij}}\rvert<\lvert{a_{ii}}\rvert\qquad\forall{i}
\end{equation*}
$$


Let's start by specifying how many rows we have in the _square_ system matrix $\mathbf{A}$ in the `number_of_rows::Int64` variable:

In [2]:
number_of_rows = 2000; # increase the number of rows for larger problem size

Next, we generate a $n\times{n}$ random system matrix $\mathbf{A}$ and a $n\times{1}$ random vector $\mathbf{b}$, [using the `randn(...)` method](https://docs.julialang.org/en/v1/stdlib/Random/#Base.randn). We add some extra to the diagonal elements of the test system matrix $\mathbf{A}$ to ensure diagonal dominance.

In [3]:
A,b = let

    # initialize -
    ϵ = 100.0; # extra stuff that we add to diagonal elements
    A = rand(number_of_rows, number_of_rows) .+ 10*ϵ*diagm(rand(number_of_rows)); # Interesting!
    b = ϵ*rand(number_of_rows);

    A,b
end;

In [4]:
A

2000×2000 Matrix{Float64}:
 319.116        0.0314316    0.779224   …    0.340884      0.328893
   0.571629   578.136        0.287021        0.973715      0.930269
   0.0885448    0.977723   123.888           0.243966      0.480077
   0.500126     0.857319     0.0818348       0.696844      0.108809
   0.0534803    0.35356      0.168584        0.276042      0.674533
   0.447574     0.242847     0.869222   …    0.308739      0.866094
   0.396669     0.260521     0.916205        0.379315      0.0492521
   0.124992     0.246376     0.669684        0.822231      0.453854
   0.598927     0.834503     0.544745        0.81007       0.309882
   0.118882     0.478093     0.819193        0.0241064     0.875559
   ⋮                                    ⋱                
   0.691551     0.265009     0.721335        0.240193      0.102888
   0.0295746    0.0863791    0.279999        0.421074      0.600043
   0.631536     0.419199     0.314859        0.595305      0.61905
   0.0262128    0.533992     0.

### Check: Is the system matrix $\mathbf{A}$ strictly diagonally dominant?
Before we continue, let's verify the randomly generated system matrix $\mathbf{A}$ is actually diagonally dominant.
> __Test:__ We'll compute the sum of the absolute values of each row (excluding the diagonal element), and compare it to the absolute value of the diagonal element in that row. If the sum of the absolute values of the non-diagonal elements is less than the absolute value of the diagonal element, then the matrix is diagonally dominant.

Is the matrix $\mathbf{A}$ strictly diagonally dominant?

In [5]:
ddcondition = let
    
    # initialize -
    ddcondition = Array{Bool,1}(undef, number_of_rows);

    # let's check each row
    for i ∈ 1:number_of_rows
        aii = abs(A[i,i]);
        σ = 0.0;
        for j ∈ 1:number_of_rows
            if (i ≠ j)
                σ += abs(A[i,j]);
            end
        end
        ddcondition[i] = (aii > σ) ? true : false; # ternary operator, nice!
    end

    ddcondition
end;

### Understanding the Diagonal Dominance Check

The `ddcondition` array contains boolean values indicating whether each row of our system matrix $\mathbf{A}$ satisfies the diagonal dominance condition. Each element `ddcondition[i]` is `true` if the absolute value of the diagonal element in row `i` is greater than the sum of the absolute values of all other elements in that row, and `false` otherwise.

> __Test:__ The assertion `@assert any(ddcondition)` checks that at least one row (and hopefully all rows) satisfies the diagonal dominance condition. If the condition fails, a warning is issued indicating that the matrix may not be suitable for iterative methods, as convergence is not guaranteed.

For diagonally dominant matrices, iterative methods like Jacobi, Gauss-Seidel, and SOR are guaranteed to converge, making them reliable choices for solving linear systems.

In [6]:
try
    @assert any(ddcondition)
catch
    @warn "Diagonal dominance condition not satisfied"
end

## Task 2: Let's test our iterative solvers
In this task, let's test the correctness of our iterative linear solver implementations. 

> __Idea:__ We'll solve the system of linear algebraic equations $\mathbf{A}\mathbf{x}=\mathbf{b}$ using the Jacobi, Gauss-Seidel, and SOR methods, and compare the results to the solution obtained using [Julia's built-in backslash operator (`\`)](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#Base.:\\-Tuple{AbstractMatrix,%20AbstractVecOrMat}), which uses a polyalgorithm approach, where the choice of a specific method depends on the problem structure and size.

First, let's use the backslash operator to obtain the Julia solution (which we will use as a reference). We'll store the reference solution the `julia_solution::Array{Float64,1}` variable.

In [7]:
julia_solution = A\b; # Julia solution

Next, let's set up a code block to compute the solutions using the Jacobi, Gauss-Seidel, and SOR methods. We'll compare the results to the Julia solution we got earlier.

Fill me in.

In [8]:
our_solution_archive = let

    # initialize -
    maximum_number_of_iterations = 1000;
    tolerance = 1e-12;
    algorithm = GaussSeidelMethod(); # change this to GaussSeidelMethod() or SuccessiveOverRelaxationMethod() to test other algorithms
    ω = 1.0; # relaxation factor, only used for SOR
    xₒ = 0.1*ones(number_of_rows); # initial solution guess xₒ

    # call the solve method with the appropriate parameters -
    x = VLDataScienceMachineLearningPackage.solve(A, b, xₒ; algorithm = algorithm, 
        maxiterations = maximum_number_of_iterations, 
        ϵ = tolerance, 
        ω = ω # only used for SOR
    );

    x; # return the solution
end;

Fill me in.

In [9]:
let

    # initialize -
    atol = 1e-8; # absolute tolerance
    i = keys(our_solution_archive) |> collect |> sort |> last; # get the last key (the last iteration)
    last_solution = our_solution_archive[i]; # get the last solution (converged solution)

    # check: our solution vs Julia solution, should be approximately equal
    @assert isapprox(julia_solution, last_solution, atol=atol)
end

## Task 3: Let's compare the performance of our solvers to the built-in Julia solvers
In this task, we'll benchmark our iterative solvers against Julia's built-in solvers to see how they perform on the same problem.

>__Expectation:__ We expect Julia's built-in solvers to be highly optimized and potentially __much__ faster than our iterative solvers, especially for larger problem sizes. However, our solvers may still perform competitively for smaller problems or specific cases.

Let's start with Julia's built-in solvers.

In [10]:
@benchmark $A\$b

BenchmarkTools.Trial: 263 samples with 1 evaluation per sample.
 Range (min … max):  16.455 ms … 41.756 ms  ┊ GC (min … max): 0.00% … 0.43%
 Time  (median):     18.358 ms              ┊ GC (median):    0.71%
 Time  (mean ± σ):   19.004 ms ±  2.421 ms  ┊ GC (mean ± σ):  0.72% ± 0.51%

          ▇█▅▁▁                                                
  ▃▁▃▁▁▄▅██████▇▇▅▃▃▃▃▁▃▃▄▂▂▁▃▂▂▁▂▂▁▁▁▁▂▂▁▁▁▂▁▃▁▁▂▂▁▁▁▁▁▁▃▁▁▂ ▃
  16.5 ms         Histogram: frequency by time        27.2 ms <

 Memory estimate: 30.56 MiB, allocs estimate: 9.

Now, let's benchmark our iterative solvers against Julia's built-in solvers to see how they perform on the same problem.

In [11]:
let
    # initialize -
    maximum_number_of_iterations = 10000;
    tolerance = 1e-12;
    algorithm = GaussSeidelMethod(); # change this to GaussSeidelMethod() or SuccessiveOverRelaxationMethod() to test other algorithms
    ω = 0.6; # relaxation factor, only used for SOR
    xₒ = 0.1*ones(number_of_rows); # initial solution guess xₒ

     # call the solve method with the appropriate parameters -
    @benchmark VLDataScienceMachineLearningPackage.solve($A, $b, $xₒ; algorithm = $algorithm, 
        maxiterations = $maximum_number_of_iterations, 
        ϵ = $tolerance, 
        ω = $ω # only used for SOR
    );
end

BenchmarkTools.Trial: 21 samples with 1 evaluation per sample.
 Range (min … max):  201.785 ms … 323.830 ms  ┊ GC (min … max): 0.25% … 23.80%
 Time  (median):     237.658 ms               ┊ GC (median):    0.37%
 Time  (mean ± σ):   247.712 ms ±  34.954 ms  ┊ GC (mean ± σ):  7.76% ± 10.43%

           ▃ ▃     █    ▃                              ▃         
  ▇▁▁▇▁▁▁▇▁█▁█▁▇▁▁▁█▇▁▁▇█▁▁▇▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▇▁▇▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▇ ▁
  202 ms           Histogram: frequency by time          324 ms <

 Memory estimate: 196.98 MiB, allocs estimate: 2669.

### What did we see?
The Julia implementation is efficient, even for larger problem sizes. The iterative solvers we implemented perform comparably (similar order of magnitude median runtime) to the built-in solvers, but there are some notable differences in memory usage and convergence behavior.

> __Differences:__ Our iterative solvers consistently use significantly more memory than the built-in solvers, particularly for larger problem sizes. Furthermore, our solvers have more allocations. This is likely due to the way we store and manipulate intermediate results (e.g., for informational purposes we keep all intermediate solutions). Occasionally, our solvers may diverge or exhibit slower convergence compared to the built-in methods.

Another data point in the ongoing story that is __buy versus build__. Unless you need some specialized behavior, it's almost always better to use established libraries and frameworks rather than building your own solutions from scratch!